In [ ]:
!pip install -q segmentation-models-pytorch torchvision

In [ ]:
import os
import shutil
import zipfile

# Clean destination and temp directories
!rm -rf /content/temp_sos /content/extracted_sos /content/data/raw/SAR_training/sentinel

#  Download dataset 
!kaggle datasets download -d bitsandlayers/sar-oil-spill-segmentation-dataset-sos -p /content/temp_sos

#  Extract ZIP
zip_files = [f for f in os.listdir("/content/temp_sos") if f.endswith(".zip")]
if not zip_files:
    raise FileNotFoundError("Download failed or zip file not found. Ensure ~/.kaggle/kaggle.json exists.")

zip_path = os.path.join("/content/temp_sos", zip_files[0])
extract_path = "/content/extracted_sos"
os.makedirs(extract_path, exist_ok=True)

print("Extracting SOS dataset...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# Define target directories
TARGET_IMG_DIR = "/content/data/raw/SAR_training/sentinel/image"
TARGET_LABEL_DIR = "/content/data/raw/SAR_training/sentinel/label"
os.makedirs(TARGET_IMG_DIR, exist_ok=True)
os.makedirs(TARGET_LABEL_DIR, exist_ok=True)


valid_exts = (".png", ".jpg", ".jpeg", ".tif", ".tiff")
copied_images = 0
copied_labels = 0

for root, dirs, files in os.walk(extract_path):
    norm_path = root.lower().replace("-", "").replace("_", "")
    
    if "sentinel" in norm_path and "train" in norm_path and "test" not in norm_path:
        if "image" in norm_path or "img" in norm_path:
            for f in files:
                if f.lower().endswith(valid_exts):
                    shutil.copy(os.path.join(root, f), os.path.join(TARGET_IMG_DIR, f))
                    copied_images += 1
        elif "label" in norm_path or "mask" in norm_path:
            for f in files:
                if f.lower().endswith(valid_exts):
                    shutil.copy(os.path.join(root, f), os.path.join(TARGET_LABEL_DIR, f))
                    copied_labels += 1

# Clean temporary files to free up disk
!rm -rf /content/temp_sos /content/extracted_sos


print("\n--- Extraction Summary ---")
print(f"Target Images: {len(os.listdir(TARGET_IMG_DIR))} (Copied: {copied_images})")
print(f"Target Labels: {len(os.listdir(TARGET_LABEL_DIR))} (Copied: {copied_labels})")

In [ ]:
import os
import sys
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split, Dataset
from PIL import Image
import torchvision.transforms.functional as TF
import segmentation_models_pytorch as smp


class SentinelDataset(Dataset):
    def __init__(self, img_dir, mask_dir, size=(256, 256)):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.size = size
        # Keep only valid image files
        valid_exts = (".png", ".jpg", ".jpeg", ".tif", ".tiff")
        self.filenames = sorted([
            f for f in os.listdir(img_dir) if f.lower().endswith(valid_exts)
        ])

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        img_name = self.filenames[idx]
        img_path = os.path.join(self.img_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name)

        image = Image.open(img_path).convert("L")
        mask = Image.open(mask_path).convert("L")

        image = TF.resize(image, self.size)
        mask = TF.resize(mask, self.size, interpolation=TF.InterpolationMode.NEAREST)

        image_tensor = TF.to_tensor(image)
        mask_tensor = TF.to_tensor(mask)
        mask_tensor = (mask_tensor > 0.5).float()

        return image_tensor, mask_tensor




BASE_DIR = os.getcwd()

# Search for the dataset path in current or parent directory
possible_paths = [
    os.path.join(BASE_DIR, "data", "raw", "SAR_training", "sentinel"),
    os.path.join(BASE_DIR, "..", "data", "raw", "SAR_training", "sentinel"),
    "/content/data/raw/SAR_training/sentinel"
]

DATA_DIR = next((p for p in possible_paths if os.path.exists(p)), possible_paths[0])
IMG_DIR = os.path.join(DATA_DIR, "image")
LABEL_DIR = os.path.join(DATA_DIR, "label")

MODEL_DIR = os.path.join(BASE_DIR, "models")
os.makedirs(MODEL_DIR, exist_ok=True)
MODEL_SAVE_PATH = os.path.join(MODEL_DIR, "sar_unet_best.pth")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



def run_training():
    if not os.path.exists(IMG_DIR) or not os.path.exists(LABEL_DIR):
        raise FileNotFoundError(
            f"Directory not found!\nImage Path: {IMG_DIR}\nLabel Path: {LABEL_DIR}"
        )

    full_dataset = SentinelDataset(IMG_DIR, LABEL_DIR)
    
    if len(full_dataset) == 0:
        raise ValueError(f"No files found in {IMG_DIR}")

    train_size = int(0.85 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_ds, val_ds = random_split(full_dataset, [train_size, val_size])

    workers = 2 if os.name != "nt" else 0
    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=workers, pin_memory=torch.cuda.is_available())
    val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=workers, pin_memory=torch.cuda.is_available())

    model = smp.Unet(encoder_name="resnet34", encoder_weights=None, in_channels=1, classes=1).to(device)
    criterion = smp.losses.DiceLoss(smp.losses.BINARY_MODE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    epochs = 15
    best_val_loss = float("inf")

    print(f"Active Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
    print(f"Total Dataset: {len(full_dataset)} images")
    print(f"Train/Val Split: {train_size} / {val_size}")
    print(f"Train Batches: {len(train_loader)} | Val Batches: {len(val_loader)}\n")
    print("Starting Training...")

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for images, masks in train_loader:
            images, masks = images.to(device), masks.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for images, masks in val_loader:
                images, masks = images.to(device), masks.to(device)
                outputs = model(images)
                val_loss += criterion(outputs, masks).item()

        train_batches = len(train_loader) if len(train_loader) > 0 else 1
        val_batches = len(val_loader) if len(val_loader) > 0 else 1

        val_loss /= val_batches
        print(f"Epoch [{epoch+1:02d}/{epochs}] | Train Loss: {total_loss/train_batches:.4f} | Val Loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            print(f"--> Saved best checkpoint to: {MODEL_SAVE_PATH}")

    print("\nModel trained successfully!")

run_training()